# 02 · Data Cleaning

This notebook focuses on cleaning and validating the datasets used in the RetailIQ Analytics project.

The cleaning process includes:

- Inspecting missing values
- Understanding the business meaning of missing values
- Handling missing product categories
- Standardizing date and timestamp columns
- Validating order lifecycle timestamps
- Creating analysis-ready derived fields
- Preserving valid records instead of blindly removing missing or unusual values

The goal is to create reliable cleaned datasets while avoiding unnecessary data loss.

In [102]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load Raw Datasets

The raw Olist datasets are loaded from the project's `data/raw` directory.

The original raw datasets are preserved and cleaned copies will be created separately.

In [103]:
DATA_PATH = Path("../data/raw")

customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    DATA_PATH / "product_category_name_translation.csv"
)

## 2. Create Cleaning Copies

The raw datasets are kept unchanged.

Separate copies are created so that cleaning operations do not modify the original raw data.

In [104]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

## 3. Missing Value Assessment

Missing values are first identified across all datasets.

Missing values will not automatically be removed or replaced. Each case will be evaluated according to its business meaning.

In [105]:
datasets = {
    "Customers": customers_clean,
    "Geolocation": geolocation_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Order Payments": order_payments_clean,
    "Order Reviews": order_reviews_clean,
    "Products": products_clean,
    "Sellers": sellers_clean,
    "Category Translation": category_translation_clean
}

for name, df in datasets.items():
    missing = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_pct": (df.isnull().mean() * 100).round(2)
    })

    missing = missing[missing["missing_count"] > 0]

    print("=" * 60)
    print(name)
    print("=" * 60)

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)
    
    print()

Customers
No missing values.

Geolocation
No missing values.

Orders
                               missing_count  missing_pct
order_approved_at                        160         0.16
order_delivered_carrier_date            1783         1.79
order_delivered_customer_date           2965         2.98

Order Items
No missing values.

Order Payments
No missing values.

Order Reviews
                        missing_count  missing_pct
review_comment_title            87656        88.34
review_comment_message          58247        58.70

Products
                            missing_count  missing_pct
product_category_name                 610         1.85
product_name_lenght                   610         1.85
product_description_lenght            610         1.85
product_photos_qty                    610         1.85
product_weight_g                        2         0.01
product_length_cm                       2         0.01
product_height_cm                       2         0.01
product_width_

## 4. Product Category Missing Values

The products dataset contains 610 products with missing category information.

Before deciding how to handle these records, we investigate whether these products actually appear in order transactions.

Dropping these products could result in loss of valid business transactions.

In [106]:
missing_category_products = products_clean[
    products_clean["product_category_name"].isnull()
]["product_id"]

missing_category_products.head(10)

105    a41e356c76fab66334f36de622ecbd3a
128    d8dee61c2034d6d075997acef1870e9b
145    56139431d72cd51f19eb9f7dae4d1617
154    46b48281eb6d663ced748f324108c733
197    5fb61f482620cb672f5e586bb132eae9
244    e10758160da97891c2fdcbc35f0f031d
294    39e3b9b12cd0bf8ee681bbc1c130feb5
299    794de06c32a626a5692ff50e4985d36f
347    7af3e2da474486a3519b0cba9dea8ad9
428    629beb8e7317703dcc5f35b5463fd20e
Name: product_id, dtype: str

In [107]:
products_clean[
    products_clean["product_category_name"].isnull()
].isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [108]:
order_items[
    order_items["product_id"].isin(missing_category_products)
].shape

(1603, 7)

### Business Impact of Missing Product Categories

The missing-category products are present in actual order transactions.

Therefore, these products should not be removed from the dataset.

We further evaluate the number of unique products, affected orders, and revenue associated with these products before deciding the appropriate treatment.

In [109]:
sold_missing_category_products = order_items[
    order_items["product_id"].isin(missing_category_products)
]["product_id"].nunique()

sold_missing_category_products

610

In [110]:
affected_orders = order_items[
    order_items["product_id"].isin(missing_category_products)
]["order_id"].nunique()

affected_orders

1451

In [111]:
affected_revenue = order_items[
    order_items["product_id"].isin(missing_category_products)
]["price"].sum()

affected_revenue

np.float64(179535.28)

### Decision: Missing Product Categories

The 610 products with missing categories are retained because they appear in 1,451 orders and represent approximately 179.5K in item revenue.

Since the actual category cannot be reliably inferred, missing category values are replaced with `unknown`.

Other missing product attributes are not filled arbitrarily because their original values cannot be reliably determined.

In [112]:
products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

In [113]:
products_clean["product_category_name"].isnull().sum()

np.int64(0)

In [114]:
products_clean["product_category_name"].value_counts().head()

product_category_name
cama_mesa_banho          3029
esporte_lazer            2867
moveis_decoracao         2657
beleza_saude             2444
utilidades_domesticas    2335
Name: count, dtype: int64

## 5. Missing Product Physical Attributes

Only a very small number of products have missing physical measurements such as weight and dimensions.

These values will be investigated before deciding whether imputation is appropriate.

In [115]:
products_clean[
    products_clean[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isnull().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Decision: Physical Attributes

Only 2 products have missing physical attributes.

Because the actual weight and dimensions cannot be reliably inferred from the available data, these values are retained as missing.

They can be excluded only from analyses that specifically require complete physical measurements.

In [116]:
products_clean[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].isnull().sum()

product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

## 6. Order Review Missing Values

The review dataset contains a large number of missing values in the optional text fields:

- `review_comment_title`
- `review_comment_message`

These fields are optional customer-provided comments.

Missing comments do not indicate invalid reviews, so the values are retained as missing rather than replaced with artificial text.

In [117]:
review_missing = pd.DataFrame({
    "missing_count": order_reviews_clean.isnull().sum(),
    "missing_pct": (
        order_reviews_clean.isnull().mean() * 100
    ).round(2)
})

review_missing[
    review_missing["missing_count"] > 0
]

,missing_count,missing_pct
review_comment_title,87656,88.34
review_comment_message,58247,58.70


## 7. Orders Data Cleaning

The orders dataset contains missing timestamps in different stages of the order lifecycle.

The order lifecycle generally follows:

Purchase → Approval → Carrier Handover → Customer Delivery

Missing timestamps are not automatically filled because their meaning depends on the order status.

For example, a cancelled or unavailable order may legitimately have no delivery date.

In [118]:
orders_clean.groupby("order_status").agg(
    total_orders=("order_id", "size"),
    missing_approved=(
        "order_approved_at",
        lambda x: x.isna().sum()
    ),
    missing_carrier=(
        "order_delivered_carrier_date",
        lambda x: x.isna().sum()
    ),
    missing_customer_delivery=(
        "order_delivered_customer_date",
        lambda x: x.isna().sum()
    )
)

,total_orders,missing_approved,missing_carrier,missing_customer_delivery
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


## 8. Delivered Order Timestamp Anomalies

Delivered orders are expected to contain the major delivery lifecycle timestamps.

Therefore, delivered orders with missing approval, carrier, or customer delivery timestamps are investigated separately.

These records are not deleted or imputed without further evidence.

In [119]:
delivered_anomalies = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (
        orders_clean["order_approved_at"].isna() |
        orders_clean["order_delivered_carrier_date"].isna() |
        orders_clean["order_delivered_customer_date"].isna()
    )
].copy()

In [120]:
delivered_anomalies[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


In [121]:
delivered_anomalies.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

## 9. Timestamp Standardization

The order timestamp columns are converted to Pandas datetime format.

This is required for:

- Date arithmetic
- Delivery duration calculation
- Chronological validation
- Monthly and yearly analysis

Invalid or unparseable timestamps are converted to missing values using `errors="coerce"`.

In [122]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

In [123]:
orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 10. Order Date Sequence Validation

The timestamps are checked against the expected order lifecycle:

Purchase → Approval → Carrier Handover → Customer Delivery

Records where a later event occurs before an earlier event are flagged as potential data-quality anomalies.

These records are flagged rather than immediately deleted because unusual timestamps may require further investigation.

In [124]:
orders_clean["invalid_date_sequence"] = (
    (
        orders_clean["order_approved_at"].notna()
        &
        (
            orders_clean["order_approved_at"]
            < orders_clean["order_purchase_timestamp"]
        )
    )
    |
    (
        orders_clean["order_approved_at"].notna()
        &
        orders_clean["order_delivered_carrier_date"].notna()
        &
        (
            orders_clean["order_delivered_carrier_date"]
            < orders_clean["order_approved_at"]
        )
    )
    |
    (
        orders_clean["order_delivered_carrier_date"].notna()
        &
        orders_clean["order_delivered_customer_date"].notna()
        &
        (
            orders_clean["order_delivered_customer_date"]
            < orders_clean["order_delivered_carrier_date"]
        )
    )
)

In [125]:
orders_clean["invalid_date_sequence"].value_counts()

invalid_date_sequence
False    98059
True      1382
Name: count, dtype: int64

In [126]:
invalid_orders = orders_clean[
    orders_clean["invalid_date_sequence"]
].copy()

In [127]:
invalid_orders[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19


In [128]:
invalid_orders["order_status"].value_counts()

order_status
delivered    1373
shipped         9
Name: count, dtype: int64

## 11. Delivery Duration

Delivery duration is calculated as the number of days between order purchase and customer delivery.

The metric is available only when a customer delivery timestamp exists.

Negative delivery duration values will be investigated separately rather than automatically removed.

In [129]:
orders_clean["delivery_time_days"] = (
    (
        orders_clean["order_delivered_customer_date"]
        - orders_clean["order_purchase_timestamp"]
    )
    .dt.total_seconds()
    / (24 * 60 * 60)
)

In [130]:
orders_clean["delivery_time_days"].describe().round(2)

count    96476.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
max        209.63
Name: delivery_time_days, dtype: float64

## 12. Cleaning Validation

After the cleaning operations, the cleaned datasets are checked again to ensure that the intended transformations were applied correctly.

For products, the category field should no longer contain missing values, while the two products with unavailable physical measurements remain missing.

In [131]:
print("Product category missing values:")
print(
    products_clean["product_category_name"]
    .isnull()
    .sum()
)

print("\nPhysical attribute missing values:")
print(
    products_clean[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isnull().sum()
)

Product category missing values:
0

Physical attribute missing values:
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64


## 13. Orders Cleaning Validation

The orders dataset is validated after timestamp standardization and delivery-duration calculation.

Missing timestamps are preserved when they represent legitimate order lifecycle states, while suspicious chronological relationships are retained as flagged anomalies for further investigation.

In [132]:
orders_clean[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_time_days",
        "invalid_date_sequence"
    ]
].head()

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,invalid_date_sequence
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,False
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,False
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,False
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,False
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,False


## 14. Data Cleaning Summary

The following cleaning decisions have been completed:

1. Raw datasets were preserved and cleaned copies were created.
2. Missing product categories were replaced with `unknown` because the affected products were part of actual transactions.
3. Missing optional review comments were retained because they represent optional customer input.
4. Missing physical product attributes were retained for the two affected products because reliable replacement values were unavailable.
5. Order timestamps were standardized to datetime format.
6. Missing order timestamps were evaluated according to order status rather than blindly imputed.
7. Order lifecycle inconsistencies were identified and flagged using `invalid_date_sequence`.
8. Delivery duration was calculated from purchase time to customer delivery time.
9. Potentially anomalous records were flagged rather than unnecessarily deleted.

The cleaned datasets are now ready for further validation, feature engineering, and exploratory data analysis.

## 15. Order Date Anomaly Investigation

The invalid date sequence flag identified potentially inconsistent order timestamps.

We now identify which specific relationship is violated:

- Approval before purchase
- Carrier handover before approval
- Customer delivery before carrier handover

This helps determine whether the records represent actual data-quality issues or valid exceptions.

In [134]:
orders_clean["approval_before_purchase"] = (
    orders_clean["order_approved_at"].notna()
    &
    (
        orders_clean["order_approved_at"]
        < orders_clean["order_purchase_timestamp"]
    )
)

orders_clean["carrier_before_approval"] = (
    orders_clean["order_approved_at"].notna()
    &
    orders_clean["order_delivered_carrier_date"].notna()
    &
    (
        orders_clean["order_delivered_carrier_date"]
        < orders_clean["order_approved_at"]
    )
)

orders_clean["delivery_before_carrier"] = (
    orders_clean["order_delivered_carrier_date"].notna()
    &
    orders_clean["order_delivered_customer_date"].notna()
    &
    (
        orders_clean["order_delivered_customer_date"]
        < orders_clean["order_delivered_carrier_date"]
    )
)

In [135]:
orders_clean[
    [
        "approval_before_purchase",
        "carrier_before_approval",
        "delivery_before_carrier"
    ]
].sum()

approval_before_purchase       0
carrier_before_approval     1359
delivery_before_carrier       23
dtype: int64

In [136]:
orders_clean[
    [
        "approval_before_purchase",
        "carrier_before_approval",
        "delivery_before_carrier"
    ]
].value_counts()

approval_before_purchase  carrier_before_approval  delivery_before_carrier
False                     False                    False                      98059
                          True                     False                       1359
                          False                    True                          23
Name: count, dtype: int64

In [137]:
orders_clean[
    orders_clean["invalid_date_sequence"]
][
    [
        "order_id",
        "order_status",
        "approval_before_purchase",
        "carrier_before_approval",
        "delivery_before_carrier"
    ]
].head(20)

,order_id,order_status,approval_before_purchase,carrier_before_approval,delivery_before_carrier
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,False,True,False
64,688052146432ef8253587b930b01a06d,delivered,False,True,False
199,58d4c4747ee059eeeb865b349b41f53a,delivered,False,True,False
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,False,True,False
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,False,True,False
481,32e4fa9bb468884309b58b9348de70c3,delivered,False,True,False
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,False,True,False
585,16e38caa92e342c7780f68832f832d4d,delivered,False,True,False
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,False,True,False
817,6051e6d3da9a50b7325cbe9c81025062,delivered,False,True,False


## 16. Order Date Anomaly Decision

The date-sequence validation identified 1,382 potentially inconsistent orders.

No records are deleted because the anomalies are concentrated in timestamp relationships and removing these orders would result in unnecessary data loss.

The anomalies are retained with an `invalid_date_sequence` flag so that they can be excluded from specific time-based analyses when required.

The main anomaly is the carrier timestamp occurring before the approval timestamp in 1,359 orders. A further 23 orders have customer delivery timestamps occurring before carrier handover.

These records will be preserved for traceability.

In [138]:
orders_clean = orders_clean.drop(
    columns=[
        "approval_before_purchase",
        "carrier_before_approval",
        "delivery_before_carrier"
    ]
)

In [139]:
orders_clean.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'invalid_date_sequence', 'delivery_time_days'],
      dtype='str')

In [140]:
orders_clean["invalid_date_sequence"].value_counts()

invalid_date_sequence
False    98059
True      1382
Name: count, dtype: int64

## 17. Duplicate Record Check

Duplicate records are checked across the cleaned datasets.

Duplicate rows can distort transaction counts, revenue calculations, and other business metrics.

Only exact duplicate rows will be considered for removal. Legitimate repeated transactions will not be removed.

In [141]:
duplicate_counts = {}

for name, df in {
    "Customers": customers_clean,
    "Geolocation": geolocation_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Order Payments": order_payments_clean,
    "Order Reviews": order_reviews_clean,
    "Products": products_clean,
    "Sellers": sellers_clean,
    "Category Translation": category_translation_clean
}.items():
    
    duplicate_counts[name] = df.duplicated().sum()

pd.Series(duplicate_counts, name="duplicate_rows")

Customers                    0
Geolocation             261831
Orders                       0
Order Items                  0
Order Payments               0
Order Reviews                0
Products                     0
Sellers                      0
Category Translation         0
Name: duplicate_rows, dtype: int64

## 18. Geolocation Duplicate Investigation

The geolocation dataset contains a large number of duplicate rows.

Unlike transactional datasets, geolocation data may legitimately contain multiple records for the same ZIP code prefix because a ZIP prefix can correspond to multiple geographic coordinates.

Therefore, duplicate rows are investigated before removal.

Only exact duplicate rows across all columns are considered redundant.

In [ ]:
geolocation_clean.duplicated().sum()

In [ ]:
geolocation_clean[
    geolocation_clean.duplicated(keep=False)
].head(10)

In [ ]:
geolocation_clean[
    geolocation_clean.duplicated()
].shape

### Geolocation Duplicate Handling

The duplicate records are exact duplicates across all five columns.

Since they do not provide additional geographic information, exact duplicate rows are removed.

Records that share the same ZIP code prefix but have different coordinates are retained because they may represent different geographic locations within the same prefix.

In [142]:
geolocation_clean = geolocation_clean.drop_duplicates().copy()

In [143]:
geolocation_clean.duplicated().sum()

np.int64(0)

In [144]:
geolocation_clean.shape

(738332, 5)

In [145]:
# verification ki kahi koi duplicate row to nhi 
geolocation_clean.duplicated().sum()

np.int64(0)

## 20. Identifier Uniqueness Check

Key identifier columns are checked for uniqueness.

Unique identifiers should not contain duplicate values in datasets where each row represents a single entity or order.

This validation helps prevent incorrect joins and duplicated records during later analysis.

In [146]:
key_checks = {
    "Customers - customer_id": customers_clean["customer_id"].duplicated().sum(),
    "Orders - order_id": orders_clean["order_id"].duplicated().sum(),
    "Products - product_id": products_clean["product_id"].duplicated().sum(),
    "Sellers - seller_id": sellers_clean["seller_id"].duplicated().sum(),
}

pd.Series(key_checks, name="duplicate_ids")

Customers - customer_id    0
Orders - order_id          0
Products - product_id      0
Sellers - seller_id        0
Name: duplicate_ids, dtype: int64

## 21. Identifier Uniqueness Validation

The primary identifiers were checked for duplicate values.

No duplicate identifiers were found in the Customers, Orders, Products, or Sellers datasets.

This confirms that these key identifiers can be safely used for joins and relationship validation in subsequent analysis.

## 22. Foreign Key Relationship Validation

Foreign key relationships are validated across the major transactional datasets.

The objective is to identify records that reference entities that do not exist in the corresponding parent dataset.

These checks help ensure that subsequent dataset merges do not unexpectedly lose records.

In [147]:
relationship_checks = {
    "Orders → Customers": (
        ~orders_clean["customer_id"].isin(
            customers_clean["customer_id"]
        )
    ).sum(),

    "Order Items → Orders": (
        ~order_items_clean["order_id"].isin(
            orders_clean["order_id"]
        )
    ).sum(),

    "Order Items → Products": (
        ~order_items_clean["product_id"].isin(
            products_clean["product_id"]
        )
    ).sum(),

    "Order Items → Sellers": (
        ~order_items_clean["seller_id"].isin(
            sellers_clean["seller_id"]
        )
    ).sum()
}

pd.Series(
    relationship_checks,
    name="orphan_records"
)

Orders → Customers        0
Order Items → Orders      0
Order Items → Products    0
Order Items → Sellers     0
Name: orphan_records, dtype: int64

## 23. Foreign Key Validation Summary

All major foreign key relationships were validated successfully.

No orphan records were found in:

- Orders → Customers
- Order Items → Orders
- Order Items → Products
- Order Items → Sellers

This confirms that the major transactional relationships are intact and the datasets can be safely joined using these identifiers.

## 24. Numeric Value Validation

Numeric columns are checked for invalid or suspicious values.

The validation focuses on business rules such as:

- Product price should not be negative
- Freight value should not be negative
- Payment value should not be negative
- Product weight and dimensions should not be negative
- Review scores should fall within the expected range
- Order item quantities should be positive

In [148]:
numeric_checks = {
    "Negative item prices": (order_items_clean["price"] < 0).sum(),
    "Negative freight values": (order_items_clean["freight_value"] < 0).sum(),
    "Negative payment values": (order_payments_clean["payment_value"] < 0).sum(),
    "Negative product weight": (products_clean["product_weight_g"] < 0).sum(),
    "Negative product length": (products_clean["product_length_cm"] < 0).sum(),
    "Negative product height": (products_clean["product_height_cm"] < 0).sum(),
    "Negative product width": (products_clean["product_width_cm"] < 0).sum(),
    "Invalid review scores": (
        ~order_reviews_clean["review_score"].between(1, 5)
    ).sum()
}

pd.Series(numeric_checks, name="invalid_values")

Negative item prices       0
Negative freight values    0
Negative payment values    0
Negative product weight    0
Negative product length    0
Negative product height    0
Negative product width     0
Invalid review scores      0
Name: invalid_values, dtype: int64

In [149]:
order_items_clean["order_item_id"].value_counts().head()

order_item_id
1    98666
2     9803
3     2287
4      965
5      460
Name: count, dtype: int64

## 25. Numeric Validation Summary

Business-related numeric fields were checked for invalid negative values and invalid review scores.

No negative prices, freight values, payment values, product measurements, or invalid review scores were found.

The `order_item_id` field contains repeated values because it represents the item sequence within an order rather than a globally unique identifier. Therefore, these repeated values are valid and are not treated as duplicates.

## 26. Order Item Uniqueness Validation

The `order_item_id` column is not globally unique because it represents the sequence of an item within an order.

Therefore, uniqueness is validated using the combination of `order_id` and `order_item_id`.

This combination should uniquely identify each item within an order.

In [150]:
order_item_key_duplicates = (
    order_items_clean
    .duplicated(
        subset=["order_id", "order_item_id"]
    )
    .sum()
)

order_item_key_duplicates

np.int64(0)

## 27. Order Item Uniqueness Summary

The combination of `order_id` and `order_item_id` was checked for duplicate records.

No duplicate combinations were found, confirming that each item within an order is uniquely represented in the dataset.

## 28. Order Payment Validation

The order payments dataset is validated using business rules for payment installments and payment types.

Payment installments should contain positive values, while payment types should belong to the expected payment categories.

In [151]:
payment_checks = {
    "Non-positive installments": (
        order_payments_clean["payment_installments"] <= 0
    ).sum(),

    "Missing payment type": (
        order_payments_clean["payment_type"].isna()
    ).sum()
}

pd.Series(payment_checks, name="invalid_payment_values")

Non-positive installments    2
Missing payment type         0
Name: invalid_payment_values, dtype: int64

In [152]:
order_payments_clean["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

## 29. Payment Installment Anomalies

Two payment records contain non-positive installment values.

These records are investigated before applying any correction. Since payment installments should normally represent the number of installments used for a payment, zero or negative values require validation rather than arbitrary replacement.

In [153]:
order_payments_clean[
    order_payments_clean["payment_installments"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [154]:
order_payments_clean[
    order_payments_clean["payment_installments"] <= 0
][
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [155]:
order_payments_clean[
    order_payments_clean["payment_type"] == "not_defined"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


## 29. Payment Anomaly Handling

Two payment records contain a non-positive installment count despite having a positive payment value.

Three records have a `not_defined` payment type with zero payment value.

Since the correct original payment information cannot be reliably inferred, these records are retained and flagged rather than deleted or arbitrarily modified.

In [156]:
order_payments_clean["invalid_installments"] = (
    order_payments_clean["payment_installments"] <= 0
)

order_payments_clean["undefined_zero_payment"] = (
    (order_payments_clean["payment_type"] == "not_defined")
    & (order_payments_clean["payment_value"] == 0)
)

In [157]:
order_payments_clean[
    [
        "invalid_installments",
        "undefined_zero_payment"
    ]
].sum()

invalid_installments      2
undefined_zero_payment    3
dtype: int64

## 30. Payment Validation Summary

Payment values and installment values were validated using business rules.

Two records were identified with zero installments despite having positive payment values.

Three records were identified with an undefined payment type and zero payment value.

No records were deleted because the original payment information cannot be reliably reconstructed. The anomalies are retained as explicit flags for traceability.

In [158]:
order_payments_clean[
    order_payments_clean["invalid_installments"]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value,invalid_installments,undefined_zero_payment
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,True,False
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,True,False


## 31. Order Reviews Validation

The order reviews dataset contains customer ratings and optional review text.

The review score has already been validated to ensure that values fall within the expected 1–5 range.

The text fields are optional, so missing review titles and messages are treated as legitimate missing information rather than errors.

We now check the review structure and identify any remaining data-quality issues.

In [159]:
review_checks = {
    "Invalid review scores": (
        ~order_reviews_clean["review_score"].between(1, 5)
    ).sum(),

    "Missing review title": (
        order_reviews_clean["review_comment_title"].isna()
    ).sum(),

    "Missing review message": (
        order_reviews_clean["review_comment_message"].isna()
    ).sum(),

    "Missing review_id": (
        order_reviews_clean["review_id"].isna()
    ).sum(),

    "Missing order_id": (
        order_reviews_clean["order_id"].isna()
    ).sum()
}

pd.Series(review_checks, name="review_checks")

Invalid review scores         0
Missing review title      87656
Missing review message    58247
Missing review_id             0
Missing order_id              0
Name: review_checks, dtype: int64

In [160]:
order_reviews_clean["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [161]:
order_reviews_clean["review_id"].duplicated().sum()

np.int64(814)

## 32. Review ID Duplicate Investigation

The review dataset contains 814 repeated `review_id` values.

Before removing these records, we investigate whether the duplicates represent completely identical rows or multiple records sharing the same review identifier.

Only exact duplicate records will be considered redundant. Different review records will be retained unless there is sufficient evidence that they are erroneous.

In [162]:
review_id_duplicates = order_reviews_clean[
    order_reviews_clean["review_id"].duplicated(keep=False)
].sort_values("review_id")

review_id_duplicates.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [163]:
review_id_duplicates[
    [
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
].head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid..."
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid..."
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...


In [164]:
order_reviews_clean.duplicated().sum()

np.int64(0)

## 33. Review Record Uniqueness

The `review_id` field contains repeated values, but the repeated records are not exact duplicates.

Some review IDs are associated with different orders. Therefore, `review_id` is not treated as a globally unique identifier.

The combination of `review_id` and `order_id` is checked instead to determine whether an individual review record is duplicated.

In [165]:
review_key_duplicates = (
    order_reviews_clean
    .duplicated(
        subset=["review_id", "order_id"]
    )
    .sum()
)

review_key_duplicates

np.int64(0)

In [166]:
order_reviews_clean.groupby("review_id")["order_id"].nunique().value_counts().sort_index()

order_id
1    97621
2      764
3       25
Name: count, dtype: int64

### Review Data Cleaning Decision

No exact duplicate review records were found.

Although 814 `review_id` values are repeated, these records represent different review-order associations and are therefore retained.

The combination of `review_id` and `order_id` is used as the review record identifier for validation purposes.

Optional review titles and messages remain missing because they represent customer choices rather than data-quality errors.

In [167]:
review_key_duplicates

np.int64(0)

In [168]:
order_reviews_clean.groupby("review_id")["order_id"].nunique().value_counts().sort_index()

order_id
1    97621
2      764
3       25
Name: count, dtype: int64

## 35. Sellers Data Validation

The sellers dataset contains seller identifiers and location information.

The dataset is validated for:
- Missing seller IDs
- Duplicate seller IDs
- Missing seller location fields
- Invalid seller identifiers

The seller ID is expected to uniquely identify each seller.

In [169]:
seller_checks = {
    "Missing seller_id": sellers_clean["seller_id"].isna().sum(),
    "Duplicate seller_id": sellers_clean["seller_id"].duplicated().sum(),
    "Missing seller_zip_code_prefix": sellers_clean["seller_zip_code_prefix"].isna().sum(),
    "Missing seller_city": sellers_clean["seller_city"].isna().sum(),
    "Missing seller_state": sellers_clean["seller_state"].isna().sum()
}

pd.Series(seller_checks, name="seller_validation")

Missing seller_id                 0
Duplicate seller_id               0
Missing seller_zip_code_prefix    0
Missing seller_city               0
Missing seller_state              0
Name: seller_validation, dtype: int64

In [170]:
sellers_clean["seller_state"].value_counts().head(10)

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64

In [171]:
sellers_clean[
    sellers_clean["seller_id"].duplicated(keep=False)
]

,seller_id,seller_zip_code_prefix,seller_city,seller_state


## 35. Sellers Validation Summary

The sellers dataset was validated for missing values and duplicate seller identifiers.

No missing seller IDs or duplicate seller IDs were found.

Seller location fields, including ZIP code, city, and state, are also complete.

The seller dataset is therefore considered structurally clean and ready for further analysis.

## 36. Category Translation Validation

The product category translation dataset maps Portuguese category names to their English equivalents.

The mapping table is checked for missing values and duplicate category names to ensure reliable category translation during analysis.

In [172]:
category_checks = {
    "Missing Portuguese category": (
        category_translation_clean["product_category_name"].isna()
    ).sum(),

    "Missing English category": (
        category_translation_clean["product_category_name_english"].isna()
    ).sum(),

    "Duplicate Portuguese category": (
        category_translation_clean["product_category_name"].duplicated()
    ).sum(),

    "Duplicate English category": (
        category_translation_clean["product_category_name_english"].duplicated()
    ).sum()
}

pd.Series(category_checks, name="category_validation")

Missing Portuguese category      0
Missing English category         0
Duplicate Portuguese category    0
Duplicate English category       0
Name: category_validation, dtype: int64

In [173]:
category_translation_clean.shape

(71, 2)

In [174]:
category_translation_clean.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


## 36. Category Translation Validation Summary

The category translation dataset contains 71 Portuguese-to-English
product category mappings.

No missing values or duplicate category names were found in either
language.

The translation mapping is therefore considered clean and reliable
for use in product category analysis.

## 37. Final Cleaned Dataset Shapes

Before saving the cleaned datasets, their final row and column counts
are reviewed to ensure that the cleaning process has not unintentionally
removed or altered records.

In [175]:
cleaned_datasets = {
    "Customers": customers_clean,
    "Geolocation": geolocation_clean,
    "Orders": orders_clean,
    "Order Items": order_items_clean,
    "Order Payments": order_payments_clean,
    "Order Reviews": order_reviews_clean,
    "Products": products_clean,
    "Sellers": sellers_clean,
    "Category Translation": category_translation_clean
}

for name, df in cleaned_datasets.items():
    print(f"{name:<22} {df.shape}")

Customers              (99441, 5)
Geolocation            (738332, 5)
Orders                 (99441, 10)
Order Items            (112650, 7)
Order Payments         (103886, 7)
Order Reviews          (99224, 7)
Products               (32951, 9)
Sellers                (3095, 4)
Category Translation   (71, 2)


## 38. Final Missing Value Audit

A final missing-value audit is performed across all cleaned datasets.

The purpose is to verify that no unexpected missing values remain after
the cleaning process and to distinguish legitimate missing information
from unresolved data-quality issues.

In [176]:
final_missing = {}

for name, df in cleaned_datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    
    final_missing[name] = missing

for name, missing in final_missing.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    
    if missing.empty:
        print("No missing values")
    else:
        print(missing)

Customers
No missing values
Geolocation
No missing values
Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
delivery_time_days               2965
dtype: int64
Order Items
No missing values
Order Payments
No missing values
Order Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64
Products
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Sellers
No missing values
Category Translation
No missing values


## 38. Final Missing Value Audit — Summary

The final audit shows that most cleaned datasets contain no missing
values.

The remaining missing values are concentrated in the Orders dataset
and are primarily related to order lifecycle timestamps.

These missing timestamps are considered meaningful rather than errors,
as different order statuses naturally have different stages of the
delivery process.

The `delivery_time_days` metric is also missing for orders where a
customer delivery timestamp is unavailable.

No arbitrary imputation is applied to these fields in order to preserve
the original business meaning of the data.

## 39. Final Duplicate Row Audit

A final duplicate-row check is performed across all cleaned datasets.

This verifies that redundant records have been removed where appropriate
while preserving valid repeated values in fields that are not globally
unique.

In [177]:
final_duplicates = {}

for name, df in cleaned_datasets.items():
    final_duplicates[name] = df.duplicated().sum()

pd.Series(
    final_duplicates,
    name="duplicate_rows"
)

Customers               0
Geolocation             0
Orders                  0
Order Items             0
Order Payments          0
Order Reviews           0
Products                0
Sellers                 0
Category Translation    0
Name: duplicate_rows, dtype: int64

## 40. Final Referential Integrity Audit

The final referential integrity audit verifies that relationships between
the major transactional datasets remain intact after cleaning.

The following relationships are validated:

- Orders → Customers
- Order Items → Orders
- Order Items → Products
- Order Items → Sellers

No orphan records should remain after the cleaning process.

In [178]:
final_orphans = {
    "Orders → Customers": (
        ~orders_clean["customer_id"].isin(
            customers_clean["customer_id"]
        )
    ).sum(),

    "Order Items → Orders": (
        ~order_items_clean["order_id"].isin(
            orders_clean["order_id"]
        )
    ).sum(),

    "Order Items → Products": (
        ~order_items_clean["product_id"].isin(
            products_clean["product_id"]
        )
    ).sum(),

    "Order Items → Sellers": (
        ~order_items_clean["seller_id"].isin(
            sellers_clean["seller_id"]
        )
    ).sum()
}

pd.Series(final_orphans, name="orphan_records")

Orders → Customers        0
Order Items → Orders      0
Order Items → Products    0
Order Items → Sellers     0
Name: orphan_records, dtype: int64

## 41. Final Data Cleaning Summary

The raw Olist datasets have been cleaned and validated for analysis.

The cleaning process included:

- Missing-value analysis and business-context investigation
- Duplicate-row detection and removal where appropriate
- Primary-key and identifier validation
- Numeric-value validation
- Date and delivery-sequence validation
- Delivery-duration calculation
- Payment-data validation
- Review-data validation
- Seller and category-translation validation
- Referential-integrity checks
- Final duplicate and missing-value audits

Important anomalies that could not be safely corrected were retained
and explicitly flagged instead of being deleted or arbitrarily modified.

Final validation confirmed:

- No duplicate rows remain in the cleaned datasets
- No orphan records exist across the major dataset relationships
- No invalid negative business values were found
- Review scores are within the valid 1–5 range
- Dataset relationships remain intact

The cleaned datasets are now ready for exploratory data analysis
and business-focused analysis.

In [179]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

file_names = {
    "Customers": "olist_customers_clean.csv",
    "Geolocation": "olist_geolocation_clean.csv",
    "Orders": "olist_orders_clean.csv",
    "Order Items": "olist_order_items_clean.csv",
    "Order Payments": "olist_order_payments_clean.csv",
    "Order Reviews": "olist_order_reviews_clean.csv",
    "Products": "olist_products_clean.csv",
    "Sellers": "olist_sellers_clean.csv",
    "Category Translation": "product_category_translation_clean.csv"
}

for name, df in cleaned_datasets.items():
    df.to_csv(processed_dir / file_names[name], index=False)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.


## 42. Verify Processed Datasets

The cleaned datasets are verified after saving to ensure that all
expected processed files have been created successfully.

In [180]:
list(processed_dir.glob("*.csv"))

[WindowsPath('../data/processed/olist_customers_clean.csv'),
 WindowsPath('../data/processed/olist_geolocation_clean.csv'),
 WindowsPath('../data/processed/olist_orders_clean.csv'),
 WindowsPath('../data/processed/olist_order_items_clean.csv'),
 WindowsPath('../data/processed/olist_order_payments_clean.csv'),
 WindowsPath('../data/processed/olist_order_reviews_clean.csv'),
 WindowsPath('../data/processed/olist_products_clean.csv'),
 WindowsPath('../data/processed/olist_sellers_clean.csv'),
 WindowsPath('../data/processed/product_category_translation_clean.csv')]

# Data Cleaning Complete

The raw Olist datasets have been thoroughly validated and cleaned.

The cleaning process covered:

- Missing-value analysis
- Duplicate detection and removal
- Primary-key validation
- Numeric-value validation
- Date and delivery validation
- Business-rule validation
- Payment anomaly detection
- Review validation
- Seller and category validation
- Referential integrity checks
- Final quality audits

All cleaned datasets have been saved to:

`data/processed/`

The processed datasets are now ready for exploratory data analysis
and business-focused analysis.